# EXP-017 원인 분석: 리랭커가 무엇을 올리고 내렸나

기준(EXP-014) 대비 정답이 상위 5위에 들어온 질의와 빠진 질의를 본다. 리랭커는 RRF 상위 20개 안의 순서만 바꾸므로, 20개 안에 정답이 있던 실패가 얼마나 살았는지와 G2 분류(D-34, 개선 가능 17·좁힐 수 없음 160)별 이동을 센다. 판정과 관계없이 쓴다. 질의 원문은 출력하지 않는다(D-31).

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.g2 import final_class
from rag.index import load_cfg

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data" / "runs"
PREFIX = "snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-k20"
PAIRS = ROOT / "data" / "rerank" / "summary_q-full-ms256-k20.pairs.jsonl"
SCORES = ROOT / "data" / "rerank" / "summary_q-full-ms256-k20-cross-encoder.scores.jsonl"
G2 = ROOT / "data" / "review" / "g2"
TOP = 5

## 질의별 이동

정답 순위는 10위 밖이면 NaN이라, 실패는 `~(순위 <= k)`로 센다.

In [7]:
cfg = load_cfg(["+exp=exp017", "index.scope=full"])
gold = load_gold(cfg, "dev-full")
rb = gold_ranks(read_run(RUNS / f"{PREFIX}-dev-full.csv"), gold, 10)
rc = gold_ranks(read_run(RUNS / f"{PREFIX}-ce-dev-full.csv"), gold, 10)
move = pd.DataFrame({k: {"진입": int((~(rb <= k) & (rc <= k)).sum()),
                         "이탈": int(((rb <= k) & ~(rc <= k)).sum())} for k in (1, 5, 10)}).T
move["순증"] = move["진입"] - move["이탈"]
move

,진입,이탈,순증
1,274,68,206
5,71,25,46
10,29,14,15


## 재정렬 후보 20개 안의 정답

재정렬은 필터 전 RRF 상위 20개에만 한다. 20개 안에 정답이 없던 실패는 리랭커로 살릴 수 없다.

In [8]:
ce = pd.DataFrame([json.loads(x) for x in SCORES.open(encoding="utf-8")])
ce["ce_rank"] = ce.groupby("qid")["score"].rank(ascending=False, method="first").astype(int)
gold_s = gold["doc_id"].rename("gold")
g = ce.merge(gold_s.rename_axis("qid").reset_index(), on="qid")
gold_ce = g[g["doc_id"] == g["gold"]].set_index("qid")["ce_rank"]
miss_b = ~(rb <= TOP)
print("질의", len(rb), ", 정답이 후보 20개 안", len(gold_ce))
print("기준 top5 실패", int(miss_b.sum()), ", 그중 후보 20개 안", int(miss_b[gold_ce.index].sum()),
      ", 그중 살아남", int((miss_b & (rc <= TOP))[gold_ce.index].sum()))

질의 3016 , 정답이 후보 20개 안 2934
기준 top5 실패 177 , 그중 후보 20개 안 95 , 그중 살아남 71


## G2 분류별 이동

기준 실패 177개의 G2 분류(`rank_type`, 개선 가능·좁힐 수 없음)마다 상위 5위에 들어온 수.

In [9]:
rows = pd.DataFrame([json.loads(x) for x in (G2 / "rows.jsonl").open(encoding="utf-8")]).set_index("qid")
s1 = {q: json.loads((G2 / "out1" / f"{q}.json").read_text(encoding="utf-8")) for q in rows.index}
s2 = {q: json.loads((G2 / "out2" / f"{q}.json").read_text(encoding="utf-8")) for q in rows.index}
rows["cls"] = [final_class(s1[q], s2[q]) for q in rows.index]
rows["gained"] = rc.reindex(rows.index) <= TOP
rows["in20"] = rows.index.isin(gold_ce.index)
pd.crosstab([rows["cls"], rows["rank_type"]], [rows["in20"], rows["gained"]], margins=True)

in20              False  True       All
gained            False False True     
cls     rank_type                      
개선 가능   결합 손실         0     2   10   12
        근접            2     1    1    4
        전 목록 실패       0     0    1    1
좁힐 수 없음 결합 손실         1     8   32   41
        근접           14    12   24   50
        전 목록 실패      65     1    3   69
All                  82    24   71  177

## 잃은 질의

기준에서 상위 5위였다가 빠진 질의의 정답이 리랭커 점수로 20개 중 몇 위였는지 본다.

In [10]:
lost = rb.index[(rb <= TOP) & ~(rc <= TOP)]
pd.DataFrame({"base": rb[lost], "cand": rc[lost], "gold_ce_rank": gold_ce.reindex(lost)}).sort_values("gold_ce_rank")

,base,cand,gold_ce_rank
qid,,,
042785-0001,4.0,6.0,6
047690-0005,1.0,6.0,6
048539-0017,1.0,6.0,6
049893-0001,5.0,6.0,6
051614-0010,1.0,6.0,6
052412-0003,1.0,6.0,6
051209-0007,5.0,6.0,6
051405-0016,1.0,6.0,6
049529-0002,3.0,7.0,7


## 정리

- 상위 5위 진입 71, 이탈 25(순증 46, Recall@5 +1.53%p). 1위는 진입 274·이탈 68로 상승이 가장 크다
- 정답이 재정렬 후보(RRF 상위 20개) 안에 있는 질의는 3,016개 중 2,934개다. 기준 실패 177개 중 후보 안 95개, 그중 71개가 살아났다. 후보 밖 82개는 리랭커로 살릴 수 없다
- G2 분류별로 개선 가능 17개 중 12개(결합 손실 12개 중 10개), 좁힐 수 없음 160개 중 59개가 진입했다. 좁힐 수 없음에서의 진입은 질의가 정답 문서를 보고 생성돼 어휘가 겹치는 점(SPEC 단서)을 cross-encoder도 이용한 결과일 수 있다(추정). 이 몫은 실사용 질의에서 줄 수 있다
- 이탈 25개는 모두 정답의 리랭커 순위가 6위 아래로 내려간 경우이고 최다 일치 필터의 영향은 없다(후보 순위 = 리랭커 순위)